# Notebook Goal

03 Bronze ingestion
```text
landing/          ← source
checkpoints/      ← stream progress
schemas/          ← Auto Loader schema tracking

Input:

Unity Catalog Volume
    └── landing/
        ├── drivers/
        ├── rides/
        └── payments/

Output(tabels):

ride_lakehouse_dev.bronze.drivers
ride_lakehouse_dev.bronze.rides
ride_lakehouse_dev.bronze.payments

JSON FILE (completed in 02)
   │
   ▼
Landing folder (completed in 02)
   │
   ▼
Auto Loader (...)
   │
   ▼
Streaming DataFrame (...)
   │
   ▼
Structured Streaming (...)
   │
   ▼
Delta Lake (...)
   │
   ▼
Bronze Table (...)
```


# Define Paths

In [0]:
BASE_PATH = "/Volumes/ride_lakehouse_dev/bronze/streaming_files"

# Incoming source files
DRIVERS_LANDING = f"{BASE_PATH}/landing/drivers"
RIDES_LANDING = f"{BASE_PATH}/landing/rides"
PAYMENTS_LANDING = f"{BASE_PATH}/landing/payments"

# Auto Loader schema tracking
DRIVERS_SCHEMA = f"{BASE_PATH}/schemas/drivers"
RIDES_SCHEMA = f"{BASE_PATH}/schemas/rides"
PAYMENTS_SCHEMA = f"{BASE_PATH}/schemas/payments"

# Structured Streaming checkpoints
DRIVERS_CHECKPOINT = f"{BASE_PATH}/checkpoints/bronze_drivers"
RIDES_CHECKPOINT = f"{BASE_PATH}/checkpoints/bronze_rides"
PAYMENTS_CHECKPOINT = f"{BASE_PATH}/checkpoints/bronze_payments"

# Driver Ingest

In [0]:


# drivers_stream = (
#     spark.readStream # this means: Treat this source incrementally.
#         .format("cloudFiles") # This invokes Auto Loader = Streaming source
#         .option("cloudFiles.format", "json") # AutoLoader still needs to knwo format
#         .option("cloudFiles.schemaLocation", DRIVERS_SCHEMA) # Store information about the detected source schema here
#         .option("cloudFiles.inferColumnTypes", "true")
#         .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # tells Auto Loader that new columns are allowed to become part of its tracked schema.
#         .load(DRIVERS_LANDING) # folder that need to be monitored
# )


In [0]:
#type(drivers_stream)

pyspark.sql.connect.dataframe.DataFrame

# AutoLoader Shema
We can even see rescued data (records that don't match your expected schema, it does not crash the streaming pipeline or silently drop the row.)

In [0]:
#drivers_stream.printSchema()

root
 |-- car_number: string (nullable = true)
 |-- experience: long (nullable = true)
 |-- generated_at: string (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
from pyspark.sql import functions as F

drivers_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", DRIVERS_SCHEMA)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(DRIVERS_LANDING)

        .withColumn(
            "_ingested_at",
            F.current_timestamp()
        )

        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )
)

In [0]:
# drivers_stream.printSchema()

root
 |-- car_number: string (nullable = true)
 |-- experience: long (nullable = true)
 |-- generated_at: string (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)



# Writing strem
Until this point, we only read stream and create streaming dataFrame(drivers_stream).

Now:
```text
readStream
   ↓
Streaming DataFrame
   ↓
writeStream
```

In [0]:
# drivers_query = (
#     drivers_stream
#         .writeStream # Write results incrementally as the streaming query processes new input

#         .format("delta") # We are using delta lake here (parquet + delta_log)

#         .outputMode("append") # it means add new batch, dont replace

#         .option(
#             "checkpointLocation",
#             DRIVERS_CHECKPOINT
#         ) # memory of streaming query, it can define which one s processed and which one is not

#         .option(
#             "mergeSchema",
#             "true"
#         ) # this is for schema evolation, to add new column for parquet file in delta

#         .trigger(
#             availableNow=True
#         ) # Process everything currently available that hasn't previously been processed, then stop. Serverless structured streaming requires this option because we dont want to run cluster all time

#         .toTable(
#             "ride_lakehouse_dev.bronze.drivers"
#         )
# )

In [0]:
# display(
#     spark.table(
#         "ride_lakehouse_dev.bronze.drivers"
#     )
# )

car_number,experience,generated_at,id,name,rating,_rescued_data,_ingested_at,_source_file
HV-2885,6,2026-09-29T11:49:13.448678+00:00,D-20260929114913448298-000,Daniel Soto,2.0,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json
QU-4530,1,2026-09-29T11:49:13.448860+00:00,D-20260929114913448298-001,Mr. Keith Yoder,1.34,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json
IR-6374,3,2026-09-29T11:49:13.449031+00:00,D-20260929114913448298-002,Courtney Douglas,1.95,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json
MV-2814,4,2026-09-29T11:49:13.449175+00:00,D-20260929114913448298-003,Charlotte Watson,1.85,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json
GF-5044,19,2026-09-29T11:49:13.449339+00:00,D-20260929114913448298-004,Courtney Morris,3.88,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json
SS-6638,17,2026-09-29T11:49:13.449484+00:00,D-20260929114913448298-005,Virginia Schneider,1.33,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json
FR-0907,4,2026-09-29T11:49:13.449623+00:00,D-20260929114913448298-006,Mary Hunt,3.61,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json
ZI-5968,16,2026-09-29T11:49:13.449762+00:00,D-20260929114913448298-007,Maria Hernandez,3.16,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json
TD-3481,19,2026-09-29T11:49:13.449899+00:00,D-20260929114913448298-008,Adam Evans,2.49,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json
RL-9618,16,2026-09-29T11:49:13.450037+00:00,D-20260929114913448298-009,Elizabeth Hernandez,2.25,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json


# Rerun streaing ingestion
Until we runing 02_generate_faker_data file again, there will be no new data in our volume and our checkpoint already record which files ingested from volume. So even we run streaming query again nothing will be processed

In [0]:
drivers_query = (
    drivers_stream
        .writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            DRIVERS_CHECKPOINT
        )
        .option(
            "mergeSchema",
            "true"
        )
        .trigger(
            availableNow=True
        )
        .toTable(
            "ride_lakehouse_dev.bronze.drivers"
        )
)

drivers_query.awaitTermination()

In [0]:
%sql
-- SELECT COUNT(*)
-- FROM ride_lakehouse_dev.bronze.drivers;

COUNT(*)
40


still 40 drivers are in our table

# Rides stream read and write

## Define rides source (autoloader)

In [0]:
rides_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", RIDES_SCHEMA)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(RIDES_LANDING)

        .withColumn(
            "_ingested_at",
            F.current_timestamp()
        )

        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )
)

## rides write (ingest to table)

In [0]:
rides_query = (
    rides_stream
        .writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            RIDES_CHECKPOINT
        )
        .option(
            "mergeSchema",
            "true"
        )
        .trigger(
            availableNow=True
        )
        .toTable(
            "ride_lakehouse_dev.bronze.rides"
        )
)

rides_query.awaitTermination()

# Payment stream read and write

## Define payment source (autoloader)

In [0]:
payments_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", PAYMENTS_SCHEMA)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(PAYMENTS_LANDING)

        .withColumn(
            "_ingested_at",
            F.current_timestamp()
        )

        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )
)

## payment write (ingest to table)

In [0]:
payments_query = (
    payments_stream
        .writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            PAYMENTS_CHECKPOINT
        )
        .option(
            "mergeSchema",
            "true"
        )
        .trigger(
            availableNow=True
        )
        .toTable(
            "ride_lakehouse_dev.bronze.payments"
        )
)

payments_query.awaitTermination()

# test

In [0]:
# tables = [
#     "ride_lakehouse_dev.bronze.drivers",
#     "ride_lakehouse_dev.bronze.rides",
#     "ride_lakehouse_dev.bronze.payments"
# ]

# for table in tables:

#     count = spark.table(table).count()

#     print(
#         f"{table}: {count} rows"
#     )

ride_lakehouse_dev.bronze.drivers: 40 rows
ride_lakehouse_dev.bronze.rides: 200 rows
ride_lakehouse_dev.bronze.payments: 200 rows


In [0]:
# display(
#     spark.table(
#         "ride_lakehouse_dev.bronze.rides"
#     ).limit(20)
# )

customer_id,distance_km,driver_id,dropoff_city,fare_amount,generated_at,pickup_city,ride_id,ride_timestamp,_rescued_data,_ingested_at,_source_file
417a1018-1529-4c27-b736-e24b03446989,9.6,D-20260929114544667600-009,Thomasburgh,25.84,2026-09-29T11:45:44.669445+00:00,West Christinehaven,R-20260929114544667600-0000,2026-09-29T11:45:44.669441+00:00,null,2026-09-29T17:38:40.553Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/rides/rides_batch_20260929114544667600.json
c0edda73-bc57-452e-a798-2d35b970b22a,3.73,D-20260929114544667600-005,Stephanieville,12.8,2026-09-29T11:45:44.669592+00:00,Lake Richardmouth,R-20260929114544667600-0001,2026-09-29T11:45:44.669590+00:00,null,2026-09-29T17:38:40.553Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/rides/rides_batch_20260929114544667600.json
7d3707c8-7ee1-4383-824f-6ca6d0de6cbe,17.44,D-20260929114544667600-001,Allenmouth,36.31,2026-09-29T11:45:44.669734+00:00,South Josephmouth,R-20260929114544667600-0002,2026-09-29T11:45:44.669732+00:00,null,2026-09-29T17:38:40.553Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/rides/rides_batch_20260929114544667600.json
8d11ad7e-037a-40c9-9ad9-5b43be8c517d,12.64,D-20260929114544667600-001,Shanestad,41.16,2026-09-29T11:45:44.669870+00:00,Chadbury,R-20260929114544667600-0003,2026-09-29T11:45:44.669868+00:00,null,2026-09-29T17:38:40.553Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/rides/rides_batch_20260929114544667600.json
28c96d5d-08fb-4ec1-b47d-1c0b6831665a,3.5,D-20260929114544667600-007,Mannborough,13.97,2026-09-29T11:45:44.670032+00:00,New Ariel,R-20260929114544667600-0004,2026-09-29T11:45:44.670029+00:00,null,2026-09-29T17:38:40.553Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/rides/rides_batch_20260929114544667600.json
ac4a3bae-5936-4fad-9cc0-8e52913fae04,14.06,D-20260929114544667600-000,North John,43.08,2026-09-29T11:45:44.670176+00:00,East Michelle,R-20260929114544667600-0005,2026-09-29T11:45:44.670173+00:00,null,2026-09-29T17:38:40.553Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/rides/rides_batch_20260929114544667600.json
c66aa5d5-9449-4a37-853b-b8fbe08124d3,11.01,D-20260929114544667600-007,West Paul,34.2,2026-09-29T11:45:44.670317+00:00,Benjaminfurt,R-20260929114544667600-0006,2026-09-29T11:45:44.670314+00:00,null,2026-09-29T17:38:40.553Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/rides/rides_batch_20260929114544667600.json
e77d99e2-2a06-4c2e-ad1a-8e317802a084,24.43,D-20260929114544667600-005,Lisaborough,50.4,2026-09-29T11:45:44.670457+00:00,Lake Kristintown,R-20260929114544667600-0007,2026-09-29T11:45:44.670454+00:00,null,2026-09-29T17:38:40.553Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/rides/rides_batch_20260929114544667600.json
793c936d-f3de-43bf-906b-7e586bdeb313,18.51,D-20260929114544667600-004,North Kevin,58.36,2026-09-29T11:45:44.670614+00:00,Ryanbury,R-20260929114544667600-0008,2026-09-29T11:45:44.670611+00:00,null,2026-09-29T17:38:40.553Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/rides/rides_batch_20260929114544667600.json
4f7dbac1-d47d-4510-816b-c4b5190faa63,7.65,D-20260929114544667600-001,South Billyside,24.23,2026-09-29T11:45:44.670762+00:00,Kathrynborough,R-20260929114544667600-0009,2026-09-29T11:45:44.670759+00:00,null,2026-09-29T17:38:40.553Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/rides/rides_batch_20260929114544667600.json


In [0]:
# display(
#     spark.table(
#         "ride_lakehouse_dev.bronze.payments"
#     ).limit(20)
# )

amount,generated_at,payment_id,payment_method,payment_timestamp,ride_id,_rescued_data,_ingested_at,_source_file
23.1,2026-09-29T11:49:16.847746+00:00,P-20260929114916836637-0000,wallet,2026-09-29T11:49:16.847744+00:00,R-20260929114916836637-0000,null,2026-09-29T17:40:07.617Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/payments/payments_batch_20260929114916836637.json
60.89,2026-09-29T11:49:16.847751+00:00,P-20260929114916836637-0001,wallet,2026-09-29T11:49:16.847749+00:00,R-20260929114916836637-0001,null,2026-09-29T17:40:07.617Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/payments/payments_batch_20260929114916836637.json
28.9,2026-09-29T11:49:16.847755+00:00,P-20260929114916836637-0002,wallet,2026-09-29T11:49:16.847753+00:00,R-20260929114916836637-0002,null,2026-09-29T17:40:07.617Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/payments/payments_batch_20260929114916836637.json
32.41,2026-09-29T11:49:16.847759+00:00,P-20260929114916836637-0003,cash,2026-09-29T11:49:16.847758+00:00,R-20260929114916836637-0003,null,2026-09-29T17:40:07.617Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/payments/payments_batch_20260929114916836637.json
70.79,2026-09-29T11:49:16.847764+00:00,P-20260929114916836637-0004,card,2026-09-29T11:49:16.847762+00:00,R-20260929114916836637-0004,null,2026-09-29T17:40:07.617Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/payments/payments_batch_20260929114916836637.json
60.07,2026-09-29T11:49:16.847769+00:00,P-20260929114916836637-0005,card,2026-09-29T11:49:16.847768+00:00,R-20260929114916836637-0005,null,2026-09-29T17:40:07.617Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/payments/payments_batch_20260929114916836637.json
10.52,2026-09-29T11:49:16.847774+00:00,P-20260929114916836637-0006,card,2026-09-29T11:49:16.847772+00:00,R-20260929114916836637-0006,null,2026-09-29T17:40:07.617Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/payments/payments_batch_20260929114916836637.json
50.61,2026-09-29T11:49:16.847778+00:00,P-20260929114916836637-0007,card,2026-09-29T11:49:16.847777+00:00,R-20260929114916836637-0007,null,2026-09-29T17:40:07.617Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/payments/payments_batch_20260929114916836637.json
18.73,2026-09-29T11:49:16.847783+00:00,P-20260929114916836637-0008,wallet,2026-09-29T11:49:16.847781+00:00,R-20260929114916836637-0008,null,2026-09-29T17:40:07.617Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/payments/payments_batch_20260929114916836637.json
77.54,2026-09-29T11:49:16.847787+00:00,P-20260929114916836637-0009,cash,2026-09-29T11:49:16.847786+00:00,R-20260929114916836637-0009,null,2026-09-29T17:40:07.617Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/payments/payments_batch_20260929114916836637.json


In [0]:
%sql
-- DESCRIBE HISTORY
-- ride_lakehouse_dev.bronze.payments;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-09-29T17:40:11.000Z,77111473424338,dogancandemir2001@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 6926ea6c-3c6a-4d84-82ab-c9db56e08e41, epochId -> 0, statsOnLoad -> true)",null,List(2906593542381095),c54895d6-9af3-4123-93f7-66f43cc35e51,0929-172807-zjmwtrkm-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 200, numOutputBytes -> 5731, numAddedFiles -> 1)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-09-29T17:40:03.000Z,77111473424338,dogancandemir2001@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""io.unitycatalog.tableId"":""b6d20936-e8d7-41e0-aae3-2efa40b558ee"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-1b3643ac-5949-4b95-8db2-dbdca18bf196"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-7eaaded4-73d4-4a6e-a82d-c998d56eeeb0""}, statsOnLoad -> false)",null,List(2906593542381095),c54895d6-9af3-4123-93f7-66f43cc35e51,0929-172807-zjmwtrkm-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
